# Fase 4 — Las 10 operaciones con Spark

**Proyecto:** BigData-Proy_Ciberseguridad · CIC-IoT-2023  
**Fase:** 4 — Procesamiento distribuido  
**Motor:** Spark

**Spark** mantiene el dataset en su propio formato de columnas en memoria y expresa cada operación como un plan de Spark que se materializa al ejecutar una acción. El plan se puede imprimir para ver la lógica antes de correrla, y las operaciones de regrouping (duplicados, `groupBy`) intercambian las particiones con un *shuffle*.

| Dato | Valor |
|---|---|
| Dataset | `01_fase1_datos/muestra/CICIoT2023_sample_600k.csv` |
| Registros | 600,000 |
| Columnas | 40 (39 numéricas + `Label`) |
| Clases | 34 (1 benigna + 33 de ataque) |
| Modo de ejecución | local, en la computadora del proyecto |
| Salida | `results/fase4/Spark/` |

Este notebook ejecuta las 10 operaciones del enunciado con **Spark** y guarda
cada resultado como CSV para poder compararlo con los demás motores.

## Las 10 operaciones del enunciado

| # | Operación | Requisito del enunciado | Archivo de salida |
|---|---|---|---|
| 01 | Carga y validación del dataset | Validación | `01_validacion.csv` |
| 02 | Limpieza de valores no válidos | Limpieza de datos | `02_limpieza.csv` |
| 03 | Tratamiento de duplicados | Eliminación de duplicados | `03_duplicados.csv` |
| 04 | Transformación de variables | Transformación de variables | `04_transformacion_variables.csv` |
| 05 | Filtrado de tráfico | Filtrado | `05_filtrado.csv` |
| 06 | Agregaciones globales | Agregaciones | `06_agregaciones.csv` |
| 07 | Agrupaciones por clase y protocolo | Agrupaciones | `07_agrupaciones.csv` |
| 08 | Ordenamiento y Top-10 | Ordenamiento | `08_ordenamiento_top10.csv` |
| 09 | Métricas de ciberseguridad | Cálculo de métricas | `09_metricas_ciberseguridad.csv` |
| 10 | Resumen consolidado por clase | CRUD y tablas de resultado | `10_resumen_consolidado.csv` |

## Reglas comunes a los cuatro motores

Para que la comparación sea justa, los cuatro notebooks aplican exactamente las
mismas reglas:

1. **Redondeo de ingesta.** Cada motor usa un parser de CSV distinto y los últimos
   decimales de un float pueden diferir en torno a `1e-11`. Todos redondean a
   **6 decimales** al leer el archivo.
2. **Valididad.** Una celda es *no válida* si está vacía, es `NaN` o es infinita.
   La operación 02 elimina las filas que contienen alguna celda no válida.
3. **Extremos.** `minimo_valido` y `maximo_valido` se calculan solo sobre valores
   finitos.
4. **Umbrales.** Los percentiles salen de `04_fase4_procesamiento/comun/umbrales.json`,
   calculado una vez con la biblioteca estándar, para que el filtro sea idéntico.
5. **Escritura.** Todos los CSV se escriben con el mismo formateador
   (`comun/io_comun.py`), así que se comparan celda por celda.

## Preparación del entorno

In [1]:
from __future__ import annotations

import os
import sys
import time
from importlib.metadata import version
from pathlib import Path

# Spark lanza procesos Python propios para ejecutar el codigo en UDFs y para
# convertir a pandas. Sin esta linea usan el Python global y fallan.
os.environ["PYSPARK_PYTHON"] = sys.executable

AQUI = Path.cwd()
FASE = AQUI if (AQUI / "comun").exists() else AQUI.parent
if str(FASE) not in sys.path:
    sys.path.insert(0, str(FASE))

from comun import config
from comun.io_comun import escribir_csv, escribir_json, pct

import pyspark
from pyspark.sql import SparkSession, functions as F, types as T

MOTOR = "spark"
VERSION = version("pyspark")
CSV_MUESTRA = config.CSV_MUESTRA
SALIDA = config.RUTA_RESULTADOS / MOTOR
SALIDA.mkdir(parents=True, exist_ok=True)
UMBRALES = config.cargar_umbrales()
TIEMPOS: list[dict] = []

# Local[*] usa todos los nucleos. En Windows Spark no puede escribir en disco
# local (falta winutils.exe), asi que los resultados se traen con collect() y
# se escriben con el formateador comun.
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("fase4-spark")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# El mayor double representable sirve para detectar infinitos: en Spark el
# lector CSV convierte "inf" en NULL, y NaN >= limite tambien es verdadero.
LIMITE_DOUBLE = float(sys.float_info.max)

COLS_ENTERAS = [
    "Protocol Type", *config.COLS_CONTADORES, "Tot sum", "Min", "Max", "Number",
]
FLOTANTES = [
    c for c in config.COLUMNAS
    if c not in COLS_ENTERAS and c != "Label"
]
ESQUEMA = T.StructType([
    T.StructField(
        c,
        T.LongType() if c in COLS_ENTERAS
        else T.StringType() if c == "Label"
        else T.DoubleType(),
        True,
    )
    for c in config.COLUMNAS
])

print(f"Spark       {pyspark.__version__}")
print(f"Python      {sys.version.split()[0]}")
print(f"Dataset     {CSV_MUESTRA.name} ({CSV_MUESTRA.stat().st_size:,} bytes)")
print(f"Resultados  {SALIDA}")


def medir(id_operacion: str, nombre: str, funcion):
    """Ejecuta la operación, cronometra y registra el tiempo."""
    inicio = time.perf_counter()
    resultado = funcion()
    segundos = round(time.perf_counter() - inicio, 3)
    TIEMPOS.append(
        {"motor": MOTOR, "id_operacion": id_operacion,
         "operacion": nombre, "segundos": segundos}
    )
    print(f"  -> [{id_operacion}] {nombre}: {segundos:.3f} s")
    return resultado


def csv(nombre: str, columnas: list[str], registros) -> None:
    """Escribe un resultado con el formateador común a los cuatro motores."""
    escribir_csv(SALIDA / nombre, columnas, registros)


def json_(nombre: str, contenido) -> None:
    escribir_json(SALIDA / nombre, contenido)


def valido(columna: str):
    """Expresión booleana: verdadero si el valor no es nulo ni infinito."""
    referencia = F.col(columna)
    if columna == "Label" or columna in COLS_ENTERAS:
        return referencia.isNotNull()
    return referencia.isNotNull() & (F.abs(referencia) < F.lit(LIMITE_DOUBLE))


def _filas(df) -> list[dict]:
    """Trae el DataFrame al driver como lista de diccionarios."""
    return [fila.asDict() for fila in df.collect()]


def tabla(filas, columnas: list[str]):
    """DataFrame de texto para mostrar un resultado en la salida del notebook.

    En estos resultados una misma celda puede ser un numero en unas filas y una
    cadena vacia en otras, y Spark no puede deducir un tipo unico para esa
    columna. Convertir todo a texto solo afecta a la vista previa: el CSV se
    sigue escribiendo con el formateador comun de `comun/io_comun.py`.
    """
    esquema = T.StructType(
        [T.StructField(columna, T.StringType(), True) for columna in columnas]
    )
    valores = [
        tuple(
            "" if v is None else str(v)
            for v in (
                [fila[c] for c in columnas]
                if isinstance(fila, dict)
                else list(fila)
            )
        )
        for fila in filas
    ]
    return spark.createDataFrame(valores, esquema)


def _tipo(columna: str) -> str:
    if columna in COLS_ENTERAS:
        return "entero"
    if columna == "Label":
        return "texto"
    return "decimal"

Spark       4.2.0
Python      3.12.3
Dataset     CICIoT2023_sample_600k.csv (124,498,206 bytes)
Resultados  C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\results\fase4\spark


## Carga del dataset

El esquema se declara de forma explícita para que Spark no tenga que deducir los tipos. El redondeo a 6 decimales se aplica con `round`, que también queda registrado en el plan.

In [2]:
def cargar():
    """Lee el CSV con el esquema declarado y normaliza la precisión."""
    datos = (
        spark.read.option("header", True)
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")
        .schema(ESQUEMA)
        .csv(str(CSV_MUESTRA))
    )
    return datos.select(
        [
            F.round(F.col(c), config.REDONDEO_INGESTA).alias(c)
            if c in FLOTANTES else F.col(c)
            for c in config.COLUMNAS
        ]
    )


df = medir("00", "Carga del dataset", cargar)

print(f"Particiones: {df.rdd.getNumPartitions()}   Columnas: {len(df.columns)}")
print(f"Filas: {df.count():,}")
df.show(5, truncate=False)

  -> [00] Carga del dataset: 1.498 s


Particiones: 16   Columnas: 40


Filas: 600,000


+-------------+-------------+------------+-----------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+---------+---------+---------+----+-----+----+------+----+---+---+---+----+----+----+----+----+----+----+-------+---+---+------+----------+--------+--------+------+------------+-----------------+
|Header_Length|Protocol Type|Time_To_Live|Rate       |fin_flag_number|syn_flag_number|rst_flag_number|psh_flag_number|ack_flag_number|ece_flag_number|cwr_flag_number|ack_count|syn_count|fin_count|rst_count|HTTP|HTTPS|DNS |Telnet|SMTP|SSH|IRC|TCP|UDP |DHCP|ARP |ICMP|IGMP|IPv |LLC |Tot sum|Min|Max|AVG   |Std       |Tot size|IAT     |Number|Variance    |Label            |
+-------------+-------------+------------+-----------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------+---------+---------+---------+----+-----+----+------+----+---+---+---+----+----+

El plan de Spark se puede inspeccionar con `explain` antes de convertir la tabla en resultado: así se ve que el redondeo forma parte del plan de lectura.

In [3]:
df.explain(True)

== Parsed Logical Plan ==
'Project ['round('Header_Length, 6) AS Header_Length#40, 'Protocol Type, 'round('Time_To_Live, 6) AS Time_To_Live#41, 'round('Rate, 6) AS Rate#42, 'round('fin_flag_number, 6) AS fin_flag_number#43, 'round('syn_flag_number, 6) AS syn_flag_number#44, 'round('rst_flag_number, 6) AS rst_flag_number#45, 'round('psh_flag_number, 6) AS psh_flag_number#46, 'round('ack_flag_number, 6) AS ack_flag_number#47, 'round('ece_flag_number, 6) AS ece_flag_number#48, 'round('cwr_flag_number, 6) AS cwr_flag_number#49, 'ack_count, 'syn_count, 'fin_count, 'rst_count, 'round('HTTP, 6) AS HTTP#50, 'round('HTTPS, 6) AS HTTPS#51, 'round('DNS, 6) AS DNS#52, 'round('Telnet, 6) AS Telnet#53, 'round('SMTP, 6) AS SMTP#54, 'round('SSH, 6) AS SSH#55, 'round('IRC, 6) AS IRC#56, 'round('TCP, 6) AS TCP#57, 'round('UDP, 6) AS UDP#58, 'round('DHCP, 6) AS DHCP#59, ... 15 more fields]
+- Relation [Header_Length#0,Protocol Type#1L,Time_To_Live#2,Rate#3,fin_flag_number#4,syn_flag_number#5,rst_flag_num

## Operación 01 — Carga y validación del dataset

**Requisito del enunciado:** Validación

Las celdas no válidas, los extremos y los valores distintos se resuelven en tres `agg` distintos: cada uno agrupa todo el dataset una sola vez, así que conviene juntarlos en uno solo para no repetir el escaneo.

In [4]:
def op01():
    total = df.count()

    # Una sola agregacion para las 40 columnas: escanear el dataset 40 veces
    # seria lo mas lento del notebook.
    expresiones = []
    for columna in config.COLUMNAS:
        if columna == "Label":
            continue
        valores = F.col(columna)
        if columna in FLOTANTES:
            finito = valores.isNotNull() & (F.abs(valores) < F.lit(LIMITE_DOUBLE))
            expresiones += [
                F.sum(F.when(~finito, 1).otherwise(0)).alias(f"{columna}__invalidas"),
                F.min(F.when(finito, valores)).alias(f"{columna}__min"),
                F.max(F.when(finito, valores)).alias(f"{columna}__max"),
            ]
        else:
            # En un entero no hay infinitos: basta con descartar los nulos.
            expresiones += [
                F.sum(F.when(valores.isNull(), 1).otherwise(0)).alias(f"{columna}__invalidas"),
                F.min(valores).alias(f"{columna}__min"),
                F.max(valores).alias(f"{columna}__max"),
            ]
    resumen = df.agg(*expresiones).first().asDict()

    # Los valores distintos solo hacen falta en un subconjunto de columnas.
    distintos = df.agg(
        *[F.countDistinct(c).alias(c) for c in config.COLS_DISTINTOS]
    ).first().asDict()
    # Polars incluye NULL (y los infinitos que Spark parsea como NULL)
    # como un valor distinto adicional.
    distintos = {
        c: int(distintos[c]) + (1 if c != "Label" and resumen[f"{c}__invalidas"] else 0)
        for c in config.COLS_DISTINTOS
    }

    registros = []
    for columna in config.COLUMNAS:
        celdas = (
            0 if columna == "Label"
            else int(resumen[f"{columna}__invalidas"])
        )
        texto = columna == "Label"
        registros.append(
            {
                "columna": columna,
                "tipo": _tipo(columna),
                "celdas_no_validas": celdas,
                "pct_no_validas": round(pct(celdas, total), 6),
                "distintos": distintos.get(columna, ""),
                "minimo_valido": "" if texto else _vacio(resumen[f"{columna}__min"]),
                "maximo_valido": "" if texto else _vacio(resumen[f"{columna}__max"]),
            }
        )
    csv("01_validacion.csv",
        ["columna", "tipo", "celdas_no_validas", "pct_no_validas", "distintos",
         "minimo_valido", "maximo_valido"],
        registros)
    json_("01_validacion_resumen.json",
          {"motor": MOTOR, "version": VERSION,
           "particiones": df.rdd.getNumPartitions(),
           "filas": total, "columnas": len(config.COLUMNAS),
           "redondeo_decimales_ingesta": config.REDONDEO_INGESTA,
           "nota": "el lector CSV de Spark convierte inf en NULL"})
    return registros


def _vacio(valor):
    """Vacio en el CSV cuando la columna no tiene ningun valor valido."""
    return "" if valor is None else valor


validacion = medir("01", "Carga y validación del dataset", op01)
tabla([tuple(f[c] for c in ["columna", "tipo", "celdas_no_validas",
                                            "pct_no_validas", "distintos",
                                            "minimo_valido", "maximo_valido"])
                       for f in validacion],
                      ["columna", "tipo", "celdas_no_validas", "pct_no_validas",
                       "distintos", "minimo_valido", "maximo_valido"]
                      ).show(12, truncate=False)

  -> [01] Carga y validación del dataset: 14.867 s


+---------------+-------+-----------------+--------------+---------+-------------+-------------+
|columna        |tipo   |celdas_no_validas|pct_no_validas|distintos|minimo_valido|maximo_valido|
+---------------+-------+-----------------+--------------+---------+-------------+-------------+
|Header_Length  |decimal|0                |0.0           |         |0.0          |60.0         |
|Protocol Type  |entero |0                |0.0           |5        |0            |47           |
|Time_To_Live   |decimal|0                |0.0           |         |0.0          |255.0        |
|Rate           |decimal|13               |0.002167      |99172    |0.000428     |7340032.0    |
|fin_flag_number|decimal|0                |0.0           |110      |0.0          |1.0          |
|syn_flag_number|decimal|0                |0.0           |145      |0.0          |1.0          |
|rst_flag_number|decimal|0                |0.0           |120      |0.0          |1.0          |
|psh_flag_number|decimal|0    

## Operación 02 — Limpieza de valores no válidos

**Requisito del enunciado:** Limpieza de datos

Un `select` sustituye cada valor no finito por `NULL` y un `dropna` descarta las filas incompletas. Son dos etapas del mismo plan, no dos jobs.

In [5]:
def op02():
    antes = df.count()
    global limpio
    limpio = df.select(
        [
            F.col(c) if c not in FLOTANTES else F.when(valido(c), F.col(c)).alias(c)
            for c in config.COLUMNAS
        ]
    ).dropna()
    despues = limpio.count()

    # Conteo de no validas de las 40 columnas en una sola agregacion.
    invalidas = df.agg(
        *[F.sum(F.when(~valido(c), 1).otherwise(0)).alias(c) for c in config.COLUMNAS]
    ).first().asDict()
    quedan = limpio.agg(
        *[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
          for c in config.COLUMNAS]
    ).first().asDict()

    registros = [
        {"columna": c,
         "valores_no_validos_antes": int(invalidas[c]),
         "valores_no_validos_despues": int(quedan[c]),
         "filas_eliminadas": antes - despues}
        for c in config.COLUMNAS if int(invalidas[c]) > 0
    ]
    csv("02_limpieza.csv",
        ["columna", "valores_no_validos_antes", "valores_no_validos_despues",
         "filas_eliminadas"],
        registros)
    json_("02_limpieza_resumen.json",
          {"motor": MOTOR,
           "criterio": "se elimina la fila con una celda vacía, NaN o infinita",
           "filas_antes": antes, "filas_despues": despues,
           "filas_eliminadas": antes - despues,
           "columnas_afectadas": registros})
    print(f"    filas antes: {antes:,}   despues: {despues:,}   "
          f"eliminadas: {antes - despues:,}")
    return limpio


limpio = medir("02", "Limpieza de valores no válidos", op02)
print(f"Filas limpias: {limpio.count():,}   Columnas: {len(limpio.columns)}")

    filas antes: 600,000   despues: 599,987   eliminadas: 13
  -> [02] Limpieza de valores no válidos: 7.979 s


Filas limpias: 599,987   Columnas: 40


## Operación 03 — Tratamiento de duplicados

**Requisito del enunciado:** Eliminación de duplicados

`dropDuplicates` resuelve las filas repetidas con un *shuffle*. Se miden las filas únicas por fila completa y por clave.

In [6]:
COLS_CLAVE = config.COLS_CLAVE_DUPLICADOS


def op03():
    antes = limpio.count()
    exactos = limpio.dropDuplicates().count()
    por_clave = limpio.dropDuplicates(COLS_CLAVE).count()

    registros = [
        {"criterio": "filas_completas",
         "columnas_clave": f"las {len(config.COLUMNAS)} columnas",
         "filas_antes": antes, "filas_despues": exactos,
         "duplicados_eliminados": antes - exactos,
         "pct_duplicados": round(pct(antes - exactos, antes), 6)},
        {"criterio": "columnas_clave",
         "columnas_clave": ", ".join(COLS_CLAVE),
         "filas_antes": antes, "filas_despues": por_clave,
         "duplicados_eliminados": antes - por_clave,
         "pct_duplicados": round(pct(antes - por_clave, antes), 6)},
    ]
    csv("03_duplicados.csv",
        ["criterio", "columnas_clave", "filas_antes", "filas_despues",
         "duplicados_eliminados", "pct_duplicados"],
        registros)

    ejemplos = (
        limpio.groupBy(*COLS_CLAVE)
        .count()
        .withColumnRenamed("count", "apariciones")
        .orderBy(F.desc("apariciones"), *[F.asc(c) for c in COLS_CLAVE])
        .limit(config.TOP_N)
    )
    csv("03_duplicados_ejemplos.csv", ["apariciones"] + COLS_CLAVE,
        _filas(ejemplos))
    return registros


duplicados = medir("03", "Tratamiento de duplicados", op03)
tabla([tuple(f.values()) for f in duplicados],
                      list(duplicados[0].keys())).show(truncate=False)

  -> [03] Tratamiento de duplicados: 8.999 s


+---------------+-------------------------------------------------+-----------+-------------+---------------------+--------------+
|criterio       |columnas_clave                                   |filas_antes|filas_despues|duplicados_eliminados|pct_duplicados|
+---------------+-------------------------------------------------+-----------+-------------+---------------------+--------------+
|filas_completas|las 40 columnas                                  |599987     |395496       |204491               |34.082572     |
|columnas_clave |Label, Protocol Type, Tot size, IAT, Rate, Number|599987     |345902       |254085               |42.348418     |
+---------------+-------------------------------------------------+-----------+-------------+---------------------+--------------+



## Operación 04 — Transformación de variables

**Requisito del enunciado:** Transformación de variables

El protocolo principal se arma encadenando `when`/`otherwise` en orden inverso al de las columnas one-hot, de modo que gana la primera que tiene un 1. Todo se resuelve en el plan sin materializar el dataset.

In [7]:
COLS_NUEVAS = ["size_kb", "rate_mbps", "coef_variacion", "total_flags",
              "protocolo_principal", "rango_iat"]

DEFINICIONES = {
    "size_kb": "Tot size / 1024 (kilobytes)",
    "rate_mbps": "Rate / 1 000 000 (paquetes por segundo)",
    "coef_variacion": "Std / AVG (dispersion del tamaño de paquete)",
    "total_flags": "Suma de los siete indicadores de flags TCP",
    "protocolo_principal": "Protocolo con valor 1 en las columnas one-hot",
    "rango_iat": "IAT clasificado con los percentiles 50 y 95: bajo, medio, alto",
}


def op04():
    iat = UMBRALES["IAT"]

    # Se recorre la lista al reves: la ultima columna procesada es la primera,
    # y por eso su when gana cuando hay varios indicadores a 1.
    protocolo = F.lit("otro")
    for columna in reversed(config.COLS_PROTOCOLO):
        protocolo = F.when(F.col(columna) > 0, F.lit(columna)).otherwise(protocolo)

    global transformado
    transformado = (
        limpio
        .withColumn("size_kb", F.col("Tot size") / F.lit(1024))
        .withColumn("rate_mbps", F.col("Rate") / F.lit(1_000_000))
        .withColumn("coef_variacion",
                    F.when(F.col("AVG") > 0, F.col("Std") / F.col("AVG")))
        .withColumn(
            "total_flags",
            F.expr(" + ".join(f"`{c}`" for c in config.COLS_FLAGS)),
        )
        .withColumn("protocolo_principal", protocolo)
        .withColumn(
            "rango_iat",
            F.when(F.col("IAT") <= F.lit(iat["p50"]), F.lit("bajo"))
            .when(F.col("IAT") <= F.lit(iat["p95"]), F.lit("medio"))
            .otherwise(F.lit("alto")),
        )
    )

    registros = []
    for columna in COLS_NUEVAS:
        texto = columna in ("protocolo_principal", "rango_iat")
        # `mean` no existe para columnas de texto en Spark, asi que las
        # estadisticas numericas solo se piden para las columnas decimales.
        if texto:
            estadisticas = transformado.agg(
                F.count(F.when(F.col(columna).isNull(), 1)).alias("nulos"),
                F.countDistinct(columna).alias("distintos"),
            ).first().asDict()
            minimo = maximo = media = ""
            distintos = int(estadisticas["distintos"])
        else:
            estadisticas = transformado.agg(
                F.count(F.when(F.col(columna).isNull(), 1)).alias("nulos"),
                F.min(columna).alias("minimo"),
                F.max(columna).alias("maximo"),
                F.mean(columna).alias("media"),
            ).first().asDict()
            minimo = estadisticas["minimo"]
            maximo = estadisticas["maximo"]
            media = estadisticas["media"]
            distintos = ""
        registros.append(
            {
                "columna": columna,
                "tipo": "texto" if texto else "decimal",
                "nulos": int(estadisticas["nulos"]),
                "minimo": minimo,
                "maximo": maximo,
                "media": media,
                "distintos": distintos,
                "definicion": DEFINICIONES[columna],
            }
        )
    csv("04_transformacion_variables.csv",
        ["columna", "tipo", "nulos", "minimo", "maximo", "media", "distintos",
         "definicion"],
        registros)

    muestra = transformado.select(
        "Label", "Protocol Type", "Rate", "Tot size", "IAT", *COLS_NUEVAS
    ).limit(config.FILAS_MUESTRA_TRANSFORMACION)
    filas = _filas(muestra)
    columnas_muestra = ["Label", "Protocol Type", "Rate", "Tot size", "IAT", *COLS_NUEVAS]
    csv("04_transformacion_muestra.csv", columnas_muestra, filas)
    return transformado, registros


transformado, variables = medir("04", "Transformación de variables", op04)
tabla([tuple(f[c] for c in
                             ["columna", "tipo", "nulos", "minimo", "maximo", "media",
                              "distintos", "definicion"]) for f in variables],
                      ["columna", "tipo", "nulos", "minimo", "maximo", "media",
                       "distintos", "definicion"]).show(10, truncate=60)

  -> [04] Transformación de variables: 11.261 s


+-------------------+-------+-----+----------------------+-----------------+-------------------+---------+------------------------------------------------------------+
|            columna|   tipo|nulos|                minimo|           maximo|              media|distintos|                                                  definicion|
+-------------------+-------+-----+----------------------+-----------------+-------------------+---------+------------------------------------------------------------+
|            size_kb|decimal|    0|           0.044921875|    4.64560546875|0.12845749221506744|         |                                 Tot size / 1024 (kilobytes)|
|          rate_mbps|decimal|    0|4.2799999999999997e-10|         7.340032|0.02851448277422554|         |                     Rate / 1 000 000 (paquetes por segundo)|
|     coef_variacion|decimal|    0|                   0.0|5.916700694160882|0.10585179549867423|         |                Std / AVG (dispersion del tamaño de pa

In [8]:
# A partir de aqui df pasa a ser el dataset limpio y transformado:
# las operaciones 05 a 10 ya pueden usar las columnas nuevas.
df = transformado
print(f"Columnas del dataset transformado: {len(df.columns)}")

Columnas del dataset transformado: 46


## Operación 05 — Filtrado de tráfico

**Requisito del enunciado:** Filtrado

Los cuatro filtros se cuentan en una sola agregación, así que el dataset se recorre una vez.

In [9]:
def op05():
    total = df.count()
    rate, size, iat = UMBRALES["Rate"], UMBRALES["Tot size"], UMBRALES["IAT"]

    filtros = [
        ("trafico_alto", f"Rate >= p95 ({rate['p95']:.6f})", rate["p95"],
         F.col("Rate") >= F.lit(rate["p95"])),
        ("paquetes_grandes", f"Tot size >= p95 ({size['p95']:.6f})", size["p95"],
         F.col("Tot size") >= F.lit(size["p95"])),
        ("trafico_intenso",
         f"Rate >= p95 ({rate['p95']:.6f}) y Tot size >= p50 ({size['p50']:.6f})",
         rate["p95"],
         (F.col("Rate") >= F.lit(rate["p95"]))
         & (F.col("Tot size") >= F.lit(size["p50"]))),
        ("iat_reducido", f"0 < IAT <= p95 ({iat['p95']:.6f})", iat["p95"],
         (F.col("IAT") > F.lit(0)) & (F.col("IAT") <= F.lit(iat["p95"]))),
    ]

    conteos = df.agg(
        *[F.sum(F.when(condicion, 1).otherwise(0)).alias(nombre)
          for nombre, _, _, condicion in filtros]
    ).first().asDict()

    registros = []
    for nombre, condicion, umbral, _ in filtros:
        encontradas = int(conteos[nombre])
        registros.append(
            {"filtro": nombre, "condicion": condicion, "umbral": umbral,
             "filas_encontradas": encontradas,
             "pct_del_total": round(pct(encontradas, total), 6)}
        )
    csv("05_filtrado.csv",
        ["filtro", "condicion", "umbral", "filas_encontradas", "pct_del_total"],
        registros)

    intenso = df.filter(
        (F.col("Rate") >= F.lit(rate["p95"]))
        & (F.col("Tot size") >= F.lit(size["p50"]))
    )
    por_label = _filas(
        intenso.groupBy("Label")
        .agg(F.count(F.lit(1)).alias("registros"),
             F.sum("Tot size").alias("volumen_bytes"),
             F.mean("Rate").alias("media_rate"))
        .orderBy(F.desc("registros"), F.asc("Label"))
    )
    csv("05_filtrado_por_label.csv",
        ["Label", "registros", "volumen_bytes", "media_rate"], por_label)
    return registros


filtrado = medir("05", "Filtrado de tráfico", op05)
tabla([tuple(f.values()) for f in filtrado],
                      list(filtrado[0].keys())).show(truncate=False)

  -> [05] Filtrado de tráfico: 4.231 s


+----------------+--------------------------------------------------------+------------+-----------------+-------------+
|filtro          |condicion                                               |umbral      |filas_encontradas|pct_del_total|
+----------------+--------------------------------------------------------+------------+-----------------+-------------+
|trafico_alto    |Rate >= p95 (63492.340297)                              |63492.340297|30029            |5.004942     |
|paquetes_grandes|Tot size >= p95 (586.680000)                            |586.68      |31235            |5.205946     |
|trafico_intenso |Rate >= p95 (63492.340297) y Tot size >= p50 (60.000000)|63492.340297|30013            |5.002275     |
|iat_reducido    |0 < IAT <= p95 (0.001070)                               |0.00107     |569989           |95.000225    |
+----------------+--------------------------------------------------------+------------+-----------------+-------------+



## Operación 06 — Agregaciones globales

**Requisito del enunciado:** Agregaciones

Las 70 métricas se piden en una sola agregación. La mediana usa `percentile`, que interpola entre los dos valores centrales, igual que NumPy o Polars (`percentile_approx` daría el valor más cercano y no coincidiría).

In [10]:
def op06():
    METRICAS = ("conteo", "suma", "media", "mediana", "minimo", "maximo",
                "desviacion_estandar")

    expresiones = []
    for columna in config.COLS_INDICADORES:
        valores = F.col(columna)
        for metrica, expresion in (
            ("conteo", F.count(valores)),
            ("suma", F.sum(valores)),
            ("media", F.mean(valores)),
            ("mediana", F.expr(f"percentile(`{columna}`, 0.5)")),
            ("minimo", F.min(valores)),
            ("maximo", F.max(valores)),
            ("desviacion_estandar", F.stddev(valores)),
        ):
            expresiones.append(expresion.alias(f"{columna}__{metrica}"))

    valores = df.agg(*expresiones).first().asDict()
    registros = [
        {"columna": columna, "metrica": metrica,
         "valor": int(valores[f"{columna}__{metrica}"])
         if metrica == "conteo" else float(valores[f"{columna}__{metrica}"])}
        for columna in config.COLS_INDICADORES
        for metrica in METRICAS
    ]
    csv("06_agregaciones.csv", ["columna", "metrica", "valor"], registros)
    return registros


agregaciones = medir("06", "Agregaciones globales", op06)
tabla([(f["columna"], f["metrica"], f["valor"]) for f in agregaciones],
                      ["columna", "metrica", "valor"]).show(14, truncate=False)

  -> [06] Agregaciones globales: 5.120 s


+--------+-------------------+------------------+
|columna |metrica            |valor             |
+--------+-------------------+------------------+
|Rate    |conteo             |599987            |
|Rate    |suma               |17108318976.259083|
|Rate    |media              |28514.482774225246|
|Rate    |mediana            |24662.221438      |
|Rate    |minimo             |0.000428          |
|Rate    |maximo             |7340032.0         |
|Rate    |desviacion_estandar|32626.85325358794 |
|Tot size|conteo             |599987            |
|Tot size|suma               |78922573.19080107 |
|Tot size|media              |131.54047202822906|
|Tot size|mediana            |60.0              |
|Tot size|minimo             |46.0              |
|Tot size|maximo             |4757.1            |
|Tot size|desviacion_estandar|228.35848631879216|
+--------+-------------------+------------------+
only showing top 14 rows


## Operación 07 — Agrupaciones por clase y protocolo

**Requisito del enunciado:** Agrupaciones

`groupBy` con las llaves `Label` y `Protocol Type` y un `orderBy` que deja el resultado ordenado sin pasos adicionales.

In [11]:
def op07():
    total = df.count()
    grupos = (
        df.groupBy("Label", "Protocol Type")
        .agg(F.count(F.lit(1)).alias("registros"),
             F.sum("Tot size").alias("volumen_bytes"),
             F.avg("Tot size").alias("media_tot_size"),
             F.avg("Rate").alias("media_rate"),
             F.avg("IAT").alias("media_iat"))
        .orderBy(F.asc("Label"), F.asc("Protocol Type"))
    )
    registros = [
        {"Label": f["Label"], "Protocol Type": f["Protocol Type"],
         "registros": int(f["registros"]),
         "pct_registros": round(pct(int(f["registros"]), total), 6),
         "volumen_bytes": f["volumen_bytes"],
         "media_tot_size": f["media_tot_size"],
         "media_rate": f["media_rate"], "media_iat": f["media_iat"]}
        for f in _filas(grupos)
    ]
    csv("07_agrupaciones.csv",
        ["Label", "Protocol Type", "registros", "pct_registros", "volumen_bytes",
         "media_tot_size", "media_rate", "media_iat"],
        registros)
    return registros


agrupaciones = medir("07", "Agrupaciones por clase y protocolo", op07)
tabla(
    [(f["Label"], f["Protocol Type"], f["registros"], f["pct_registros"],
      f["volumen_bytes"], f["media_tot_size"], f["media_rate"], f["media_iat"])
     for f in agrupaciones],
    ["Label", "Protocol Type", "registros", "pct_registros", "volumen_bytes",
     "media_tot_size", "media_rate", "media_iat"]).show(10, truncate=False)

  -> [07] Agrupaciones por clase y protocolo: 2.663 s


+----------------+-------------+---------+-------------+------------------+------------------+------------------+--------------------+
|Label           |Protocol Type|registros|pct_registros|volumen_bytes     |media_tot_size    |media_rate        |media_iat           |
+----------------+-------------+---------+-------------+------------------+------------------+------------------+--------------------+
|Backdoor_Malware|6            |28       |0.004667     |10134.800000000001|361.9571428571429 |673.4271656071429 |0.026104821428571425|
|Backdoor_Malware|17           |14       |0.002333     |2075.0            |148.21428571428572|48.06514835714286 |0.03846371428571428 |
|Benign          |0            |31       |0.005167     |4701.799999999999 |151.67096774193547|221.48169361290317|0.01910541935483871 |
|Benign          |1            |3        |0.0005       |408.79999999999995|136.26666666666665|128.23709366666668|0.018639            |
|Benign          |6            |12885    |2.147547     

## Operación 08 — Ordenamiento y Top-10

**Requisito del enunciado:** Ordenamiento

Resumen por clase ordenado por volumen descendente. Spark ordena dentro de cada partición y luego mezcla, de modo que el `limit` no necesita traer las 34 filas.

In [12]:
def _resumen_por_clase():
    """Volumen y media de tasa por clase, ordenado por volumen."""
    return _filas(
        df.groupBy("Label")
        .agg(F.count(F.lit(1)).alias("registros"),
             F.sum("Tot size").alias("volumen_bytes"),
             F.avg("Rate").alias("media_rate"))
        .orderBy(F.desc("volumen_bytes"), F.asc("Label"))
    )


def op08():
    volumen_total = float(df.agg(F.sum("Tot size")).first()[0])
    top = _resumen_por_clase()[:config.TOP_N]
    registros = [
        {"posicion": posicion, "Label": f["Label"], "registros": int(f["registros"]),
         "volumen_bytes": f["volumen_bytes"],
         "volumen_mb": f["volumen_bytes"] / (1024 * 1024),
         "pct_volumen": round(pct(float(f["volumen_bytes"]), volumen_total), 6),
         "media_rate": f["media_rate"]}
        for posicion, f in enumerate(top, start=1)
    ]
    csv("08_ordenamiento_top10.csv",
        ["posicion", "Label", "registros", "volumen_bytes", "volumen_mb",
         "pct_volumen", "media_rate"],
        registros)
    return registros


top10 = medir("08", "Ordenamiento y Top-10", op08)
tabla([(f["posicion"], f["Label"], f["registros"], f["volumen_bytes"],
                        f["volumen_mb"], f["pct_volumen"], f["media_rate"])
                       for f in top10],
                      ["posicion", "Label", "registros", "volumen_bytes", "volumen_mb",
                       "pct_volumen", "media_rate"]).show(truncate=False)

  -> [08] Ordenamiento y Top-10: 3.266 s


+--------+-----------------------+---------+------------------+------------------+-----------+------------------+
|posicion|Label                  |registros|volumen_bytes     |volumen_mb        |pct_volumen|media_rate        |
+--------+-----------------------+---------+------------------+------------------+-----------+------------------+
|1       |Benign                 |14085    |8575966.255555985 |8.178678756290422 |10.866303  |2667.4811567913343|
|2       |Mirai-greeth_flood     |12719    |7433444.719666992 |7.089085311572068 |9.418655   |5576.009272484942 |
|3       |Mirai-udpplain         |11423    |6243729.962310004 |5.954484903631214 |7.911209   |6132.884517090259 |
|4       |DDoS-ICMP_Flood        |92356    |5610000.217274016 |5.350113122247711 |7.108233   |39957.26767816771 |
|5       |Mirai-greip_flood      |9642     |5451961.8560679965|5.1993959961585965|6.907988   |5055.603641788217 |
|6       |DDoS-ICMP_Fragmentation|5804     |5140472.465007001 |4.902336564070702 |6.5133

## Operación 09 — Métricas de ciberseguridad

**Requisito del enunciado:** Cálculo de métricas

Indicadores de población, volumen, comportamiento de flags, concentración y calidad del dataset.

In [13]:
def op09():
    base = df.agg(
        F.count(F.lit(1)).alias("total"),
        F.sum(F.when(F.col("Label") == config.ETIQUETA_BENIGNA, 1)
              .otherwise(0)).alias("benignos"),
        F.countDistinct("Label").alias("clases"),
        F.countDistinct(F.when(F.col("Label") != config.ETIQUETA_BENIGNA,
                               F.col("Label"))).alias("clases_ataque"),
        F.sum("Tot size").alias("volumen"),
        F.sum("Number").alias("paquetes"),
        F.avg("syn_flag_number").alias("media_syn"),
        F.avg("ack_flag_number").alias("media_ack"),
        F.avg("rst_flag_number").alias("media_rst"),
        F.avg("fin_flag_number").alias("media_fin"),
        F.avg("IAT").alias("iat_medio"),
        F.avg("Rate").alias("tasa_media"),
    ).first().asDict()
    total = int(base["total"])
    benignos = int(base["benignos"])
    ataques = total - benignos
    volumen_total = float(base["volumen"])
    paquetes = float(base["paquetes"])
    duplicados = total - df.dropDuplicates().count()

    por_clase = _resumen_por_clase()
    principal = por_clase[0]
    top5 = por_clase[:5]
    volumen_top5 = sum(float(f["volumen_bytes"]) for f in top5)
    registros_top5 = sum(int(f["registros"]) for f in top5)

    metricas = [
        ("poblacion", "total_registros", total),
        ("poblacion", "total_clases", int(base["clases"])),
        ("poblacion", "clases_de_ataque", int(base["clases_ataque"])),
        ("poblacion", "registros_benignos", benignos),
        ("poblacion", "registros_de_ataque", ataques),
        ("poblacion", "pct_registros_benignos", round(pct(benignos, total), 6)),
        ("poblacion", "pct_registros_ataque", round(pct(ataques, total), 6)),
        ("volumen", "volumen_total_bytes", volumen_total),
        ("volumen", "volumen_total_mb", volumen_total / (1024 * 1024)),
        ("volumen", "paquetes_totales", paquetes),
        ("volumen", "tamano_medio_paquete_bytes",
         volumen_total / paquetes if paquetes else 0.0),
        ("volumen", "tasa_media", float(base["tasa_media"])),
        ("comportamiento", "media_syn_flag", float(base["media_syn"])),
        ("comportamiento", "media_ack_flag", float(base["media_ack"])),
        ("comportamiento", "media_rst_flag", float(base["media_rst"])),
        ("comportamiento", "media_fin_flag", float(base["media_fin"])),
        ("comportamiento", "ratio_syn_ack",
         float(base["media_syn"]) / float(base["media_ack"])
         if base["media_ack"] else 0.0),
        ("comportamiento", "iat_medio", float(base["iat_medio"])),
        ("concentracion", "clase_principal", principal["Label"]),
        ("concentracion", "pct_registros_clase_principal",
         round(pct(int(principal["registros"]), total), 6)),
        ("concentracion", "pct_registros_top5",
         round(pct(registros_top5, total), 6)),
        ("concentracion", "pct_volumen_top5",
         round(pct(volumen_top5, volumen_total), 6)),
        ("calidad", "filas_duplicadas", duplicados),
        ("calidad", "pct_filas_duplicadas", round(pct(duplicados, total), 6)),
        ("calidad", "pct_filas_unicas", round(pct(total - duplicados, total), 6)),
    ]
    registros = [{"categoria": c, "metrica": m, "valor": v} for c, m, v in metricas]
    csv("09_metricas_ciberseguridad.csv", ["categoria", "metrica", "valor"], registros)
    return registros


metricas = medir("09", "Métricas de ciberseguridad", op09)
tabla([(f["categoria"], f["metrica"], str(f["valor"])) for f in metricas],
                      ["categoria", "metrica", "valor"]).show(25, truncate=False)

  -> [09] Métricas de ciberseguridad: 7.742 s


+--------------+-----------------------------+--------------------+
|categoria     |metrica                      |valor               |
+--------------+-----------------------------+--------------------+
|poblacion     |total_registros              |599987              |
|poblacion     |total_clases                 |34                  |
|poblacion     |clases_de_ataque             |33                  |
|poblacion     |registros_benignos           |14085               |
|poblacion     |registros_de_ataque          |585902              |
|poblacion     |pct_registros_benignos       |2.347551            |
|poblacion     |pct_registros_ataque         |97.652449           |
|volumen       |volumen_total_bytes          |78922573.19080107   |
|volumen       |volumen_total_mb             |75.26643103675944   |
|volumen       |paquetes_totales             |57290468.0          |
|volumen       |tamano_medio_paquete_bytes   |1.3775864632629824  |
|volumen       |tasa_media                   |28

## Operación 10 — Resumen consolidado por clase

**Requisito del enunciado:** CRUD y tablas de resultado

Tabla final por clase con el conteo, el peso porcentual, el volumen y las medias de las variables transformadas.

In [14]:
def op10():
    total = df.count()
    volumen_total = float(df.agg(F.sum("Tot size")).first()[0])
    resumen = _filas(
        df.groupBy("Label")
        .agg(F.count(F.lit(1)).alias("registros"),
             F.sum("Tot size").alias("volumen_bytes"),
             F.avg("Rate").alias("media_rate"),
             F.avg("Tot size").alias("media_tot_size"),
             F.avg("size_kb").alias("media_size_kb"),
             F.avg("IAT").alias("media_iat"),
             F.avg("syn_flag_number").alias("media_syn"),
             F.avg("ack_flag_number").alias("media_ack"),
             F.avg("rst_flag_number").alias("media_rst"),
             F.avg("total_flags").alias("media_total_flags"),
             F.countDistinct("Protocol Type").alias("protocolos_distintos"))
        .orderBy(F.desc("registros"), F.asc("Label"))
    )
    registros = [
        {"posicion": posicion, "Label": f["Label"], "registros": int(f["registros"]),
         "pct_registros": round(pct(int(f["registros"]), total), 6),
         "volumen_bytes": f["volumen_bytes"],
         "pct_volumen": round(pct(float(f["volumen_bytes"]), volumen_total), 6),
         "media_rate": f["media_rate"], "media_tot_size": f["media_tot_size"],
         "media_size_kb": f["media_size_kb"], "media_iat": f["media_iat"],
         "media_syn": f["media_syn"], "media_ack": f["media_ack"],
         "media_rst": f["media_rst"], "media_total_flags": f["media_total_flags"],
         "protocolos_distintos": int(f["protocolos_distintos"])}
        for posicion, f in enumerate(resumen, start=1)
    ]
    csv("10_resumen_consolidado.csv",
        ["posicion", "Label", "registros", "pct_registros", "volumen_bytes",
         "pct_volumen", "media_rate", "media_tot_size", "media_size_kb", "media_iat",
         "media_syn", "media_ack", "media_rst", "media_total_flags",
         "protocolos_distintos"],
        registros)
    return registros


consolidado = medir("10", "Resumen consolidado por clase", op10)
tabla(
    [(f["posicion"], f["Label"], f["registros"], f["pct_registros"], f["volumen_bytes"],
      f["pct_volumen"], f["media_rate"], f["media_tot_size"]) for f in consolidado],
    ["posicion", "Label", "registros", "pct_registros", "volumen_bytes",
     "pct_volumen", "media_rate", "media_tot_size"]).show(10, truncate=False)

  -> [10] Resumen consolidado por clase: 4.339 s


+--------+-----------------------+---------+-------------+------------------+-----------+------------------+------------------+
|posicion|Label                  |registros|pct_registros|volumen_bytes     |pct_volumen|media_rate        |media_tot_size    |
+--------+-----------------------+---------+-------------+------------------+-----------+------------------+------------------+
|1       |DDoS-ICMP_Flood        |92356    |15.393       |5610000.217274013 |7.108233   |39957.267678167715|60.743213405452956|
|2       |DDoS-UDP_Flood         |69419    |11.570084    |4218631.755966996 |5.345279   |33345.355260156415|60.77056362043527 |
|3       |DDoS-TCP_Flood         |57687    |9.614708     |3631695.7648180076|4.601593   |32271.1515846533  |62.955185133877784|
|4       |DDoS-PSHACK_FLOOD      |52521    |8.75369      |3175766.8133120066|4.023902   |32611.295327694093|60.46660980011818 |
|5       |DDoS-SYN_Flood         |52065    |8.677688     |3247846.6277010143|4.115232   |28803.489590682

## Resumen de la ejecución

La tabla muestra el tiempo de cada operación. El total incluye la carga del CSV.

In [15]:
total = round(sum(f["segundos"] for f in TIEMPOS), 3)
ancho = max(len(f["operacion"]) for f in TIEMPOS)
for f in TIEMPOS:
    print(f"  {f['id_operacion']}  {f['operacion']:<{ancho}}  {f['segundos']:>8.3f} s")
print(f"  Total: {total:.3f} s")

csv("00_tiempos.csv", ["motor", "id_operacion", "operacion", "segundos"], TIEMPOS)
json_(
    "00_resumen_ejecucion.json",
    {"motor": MOTOR, "version": VERSION, "dataset": CSV_MUESTRA.name,
     "maestro": spark.sparkContext.master,
     "particiones": spark.sparkContext.defaultParallelism,
     "filas_cargadas": limpio.count(), "columnas": len(config.COLUMNAS),
     "operaciones": 10, "segundos_totales": total, "tiempos": TIEMPOS,
     "nota": "en Windows Spark no puede escribir a disco local: los resultados se "
             "traen con collect() y se escriben con el formateador comun"},
)
print(f"Archivos escritos en: {SALIDA}")

  00  Carga del dataset                      1.498 s
  01  Carga y validación del dataset        14.867 s
  02  Limpieza de valores no válidos         7.979 s
  03  Tratamiento de duplicados              8.999 s
  04  Transformación de variables           11.261 s
  05  Filtrado de tráfico                    4.231 s
  06  Agregaciones globales                  5.120 s
  07  Agrupaciones por clase y protocolo     2.663 s
  08  Ordenamiento y Top-10                  3.266 s
  09  Métricas de ciberseguridad             7.742 s
  10  Resumen consolidado por clase          4.339 s
  Total: 71.965 s


Archivos escritos en: C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\results\fase4\spark


## Cierre de la sesión

Se detiene el `SparkSession` para liberar la memoria antes de seguir.

In [16]:
spark.stop()
print('Spark detenido.')

Spark detenido.
